# 🎯 YOLO26n — Bayesian Hyperparameter Sweep (W&B Sweeps)
> **Pre-Training Optimization** — finds optimal `lr0`, `lrf`, `momentum`, `weight_decay`, and `warmup_epochs` before committing to a full 50-epoch training run.

| Setting | Value | Rationale |
|---|---|---|
| Strategy | Bayesian (Gaussian Process) | Uses past trials to predict the most promising next config |
| Trials | 15 | First ≈6 are random warm-up, the rest are model-guided |
| Epochs / trial | 8 | Fast proxy on 32% subset — enough to reliably rank configs |
| Optimizer | **AdamW (fixed)** | See ⚠️ note in Section 2 — critical for sweep validity |
| Subset | Stratified 32% | All 24 classes guaranteed to appear in every micro-trial |

---


## 📦 Section 1 — Installation & Imports

In [1]:
!pip install -q ultralytics wandb pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 109.1 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[

In [2]:
import subprocess
#subprocess.run(["pip", "install", "-qU", "ultralytics", "wandb", "pyyaml"], check=True)

# ── Standard library ──────────────────────────────────────────────────────────
import os, gc, shutil, random, json
from pathlib import Path
from typing import Dict, List, Optional

# ── Third-party ───────────────────────────────────────────────────────────────
import yaml
import torch
import wandb
from ultralytics import YOLO

print(f"✅ Imports OK | torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Imports OK | torch 2.10.0+cu128 | CUDA: True


### Merging 2 datasets from Kaggle

In [3]:
# ── Paste your exact paths here ──
OUTPUT_PATH = "/kaggle/working/v2_acc_and_fire_datasets_merged"

ACCIDENT_TRAIN_IMAGES = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/train/images"   # change
ACCIDENT_TRAIN_LABELS = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/train/labels"   # change
ACCIDENT_VAL_IMAGES   = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/valid/images"   # change
ACCIDENT_VAL_LABELS   = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/valid/labels"   # change
ACCIDENT_TEST_IMAGES  = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/test/images"    # change
ACCIDENT_TEST_LABELS  = "/kaggle/input/datasets/maryamsamirelsayed/accident-types-and-detection-dataset/test/labels"    # change

FIRE_TRAIN_IMAGES = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/images"  # change
FIRE_TRAIN_LABELS = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/train/labels"  # change
FIRE_VAL_IMAGES   = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/images"    # change
FIRE_VAL_LABELS   = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/val/labels"    # change
FIRE_TEST_IMAGES  = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/images"   # change
FIRE_TEST_LABELS  = "/kaggle/input/datasets/sayedgamal99/smoke-fire-detection-yolo/data/test/labels"   # change

# ── Merged class list ──
CLASSES = [
    'bus', 'bus_bus_accident', 'bus_object_accident', 'bus_person_accident',
    'bus_truck_accident', 'car', 'car_bus_accident', 'car_car_accident',
    'car_motorcycle_accident', 'car_object_accident', 'car_person_accident',
    'car_truck_accident', 'motorcycle', 'motorcycle_bus_accident',
    'motorcycle_motorcycle_accident', 'motorcycle_object_accident',
    'motorcycle_person_accident', 'motorcycle_truck_accident', 'truck',
    'truck_object_accident', 'truck_person_accident', 'truck_truck_accident',
    'smoke', 'fire'
]

# ── Create output folders ──
for split in ['train', 'val', 'test']:
    os.makedirs(f"{OUTPUT_PATH}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUTPUT_PATH}/labels/{split}", exist_ok=True)

# ── Copy function ──
def copy_split(img_dir, lbl_dir, prefix, class_offset, split):
    if not os.path.exists(img_dir):
        print(f"  Skipping {split} — folder not found: {img_dir}")
        return
    count = 0
    for fname in os.listdir(img_dir):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        stem = os.path.splitext(fname)[0]
        shutil.copy(f"{img_dir}/{fname}", f"{OUTPUT_PATH}/images/{split}/{prefix}_{fname}")
        lbl_src = f"{lbl_dir}/{stem}.txt"
        lbl_dst = f"{OUTPUT_PATH}/labels/{split}/{prefix}_{stem}.txt"
        if os.path.exists(lbl_src):
            with open(lbl_src) as f:
                lines = f.readlines()
            with open(lbl_dst, 'w') as f:
                for line in lines:
                    parts = line.strip().split()
                    if parts:
                        parts[0] = str(int(parts[0]) + class_offset)
                        f.write(' '.join(parts) + '\n')
        count += 1
    print(f"  [{split}] {prefix}: {count} images copied")

# ── Run merge ──
print("Copying accident dataset...")
copy_split(ACCIDENT_TRAIN_IMAGES, ACCIDENT_TRAIN_LABELS, 'acc', class_offset=0,  split='train')
copy_split(ACCIDENT_VAL_IMAGES,   ACCIDENT_VAL_LABELS,   'acc', class_offset=0,  split='val')
copy_split(ACCIDENT_TEST_IMAGES,  ACCIDENT_TEST_LABELS,  'acc', class_offset=0,  split='test')

print("Copying fire dataset...")
copy_split(FIRE_TRAIN_IMAGES, FIRE_TRAIN_LABELS, 'fire', class_offset=22, split='train')
copy_split(FIRE_VAL_IMAGES,   FIRE_VAL_LABELS,   'fire', class_offset=22, split='val')
copy_split(FIRE_TEST_IMAGES,  FIRE_TEST_LABELS,  'fire', class_offset=22, split='test')

# ── Save data.yaml ──
with open(f"{OUTPUT_PATH}/data.yaml", 'w') as f:
    f.write(f"path: {OUTPUT_PATH}\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("test: images/test\n")
    f.write(f"nc: {len(CLASSES)}\n")
    f.write(f"names: {CLASSES}\n")

# ── Summary ──
print("\n✅ Merge complete!")
for split in ['train', 'val', 'test']:
    n = len(os.listdir(f"{OUTPUT_PATH}/images/{split}"))
    print(f"  {split}: {n} images")
print(f"  Total classes: {len(CLASSES)}")
print(f"  Saved to: {OUTPUT_PATH}")

Copying accident dataset...
  [train] acc: 23808 images copied
  [val] acc: 2501 images copied
  [test] acc: 1245 images copied
Copying fire dataset...
  [train] fire: 14122 images copied
  [val] fire: 3099 images copied
  [test] fire: 4306 images copied

✅ Merge complete!
  train: 37930 images
  val: 5600 images
  test: 5551 images
  Total classes: 24
  Saved to: /kaggle/working/v2_acc_and_fire_datasets_merged


## ⚙️ Section 2 — Configuration

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# All tuneable constants live here. Edit this cell before running the sweep.
# ═══════════════════════════════════════════════════════════════════════════════

# ── Paths ──────────────────────────────────────────────────────────────────────
MERGED_DATASET_PATH = "/kaggle/working/v2_acc_and_fire_datasets_merged"
SUBSET_PATH         = "/kaggle/working/sweep_subset"
SUBSET_YAML_PATH    = str(Path(SUBSET_PATH) / "subset_data.yaml")
SWEEP_RESULTS_PATH  = "/kaggle/working/sweep_trial_results"

# ── Base model ─────────────────────────────────────────────────────────────────
# Use your previously fine-tuned last.pt as the starting checkpoint.
# This gives micro-trials dynamics representative of your actual fine-tuning,
# so the ranked hyperparameters will be more valid for your full training run.
# ▶  Switch to "yolo26n.pt" only if you want to search from scratch.
#BASE_MODEL = ("safespace-final-models/pytorch/default/9/last.pt")
BASE_MODEL = "yolo26n.pt"   # ← uncomment for from-scratch search

# ── Sweep budget ───────────────────────────────────────────────────────────────
NUM_TRIALS   = 15   # total Bayesian trials (first ~6 act as random warm-up)
MICRO_EPOCHS = 8    # epochs per trial — fast proxy for final performance ranking

# ── Stratified subset strategy ─────────────────────────────────────────────────
# Your 24-class dataset has SEVERE imbalance:
#   Common classes : car(1240), motorcycle(994), truck(931), smoke(1550), fire(879)
#   Rare classes   : motorcycle_truck_accident(37), bus_object_accident(32) ...
#
# A naive random 10% sample would leave rare accident classes with only 3-6 images,
# making per-class mAP50 noisy and the Bayesian optimizer signals unreliable.
#
# Strategy:
#   • Classes with < RARE_THRESHOLD images → keep ALL  (guarantees all 24 classes)
#   • Classes with ≥ RARE_THRESHOLD images → keep SUBSET_FRAC (32%) randomly
RARE_THRESHOLD = 2000   # images below this → keep entire class 
SUBSET_FRAC    = 0.32  # fraction of common (>= RARE_THRESHOLD) classes to keep

# ── Fixed (non-swept) training hyperparameters ────────────────────────────────
# These are taken directly from your actual training logs to keep micro-trials
# as representative of full training as possible.
FIXED_BATCH         = 64      # from logs: batch=16
FIXED_IMGSZ         = 640     # from logs: imgsz=640
FIXED_COS_LR        = True   # from logs: cos_lr=False  (standard LR scheduler)
FIXED_AMP           = True    # from logs: amp=True      (mixed precision)
FIXED_WORKERS       = 8       # from logs: workers=8
FIXED_DETERMINISTIC = True    # from logs: deterministic=True
FIXED_PATIENCE      = 999     # disabled — no early stopping during micro-trials

# ── ⚠️  CRITICAL — Optimizer decision ────────────────────────────────────────
# Your training config uses optimizer="auto". With YOLO26, "auto" selects:
#
#   Micro-trial  (this sweep) : ~560  imgs / 16 batch × 5  ep ≈    175 iters → AdamW
#   Full training (50 epochs) : 5600  imgs / 16 batch × 50 ep ≈ 17,500 iters → MuSGD ⚠️
#
# If optimizer="auto" were used in both, the sweep would tune AdamW hyperparameters
# (lr0, momentum) but your full run would apply them to MuSGD — making the
# swept values INVALID for the final training run.
#
# Fix: we lock optimizer="AdamW" for ALL micro-trials here, AND you should
# update your final training notebook to use optimizer="AdamW" when applying results.
FIXED_OPTIMIZER = "MuSGD"

# ── W&B settings ──────────────────────────────────────────────────────────────
WB_PROJECT    = "HyperParameters_Tuning_Fire_Plus_Accidents"    
WB_GROUP      = "bayesian_hp_sweep"      # groups all 15 trials in W&B UI
WB_JOB_TYPE   = "sweep_trial"
WB_SWEEP_NAME = "yolo26n_bayes_sweep"

# ── 24-class list (must match your data.yaml exactly) ─────────────────────────
CLASSES = [
    "bus",                             "bus_bus_accident",
    "bus_object_accident",             "bus_person_accident",
    "bus_truck_accident",              "car",
    "car_bus_accident",                "car_car_accident",
    "car_motorcycle_accident",         "car_object_accident",
    "car_person_accident",             "car_truck_accident",
    "motorcycle",                      "motorcycle_bus_accident",
    "motorcycle_motorcycle_accident",  "motorcycle_object_accident",
    "motorcycle_person_accident",      "motorcycle_truck_accident",
    "truck",                           "truck_object_accident",
    "truck_person_accident",           "truck_truck_accident",
    "smoke",                           "fire",
]
NUM_CLASSES = len(CLASSES)   # 24

print(
    f"✅ Config loaded | {NUM_CLASSES} classes | "
    f"{NUM_TRIALS} trials × {MICRO_EPOCHS} epochs | "
    f"optimizer={FIXED_OPTIMIZER} | model={Path(BASE_MODEL).name}"
)


✅ Config loaded | 24 classes | 15 trials × 8 epochs | optimizer=MuSGD | model=yolo26n.pt


## 🔐 Section 3 — W&B Authentication

In [5]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WandB_SafeSpace")

wandb.login(key=wandb_api_key)

print("✅ W&B authenticated")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ahmed-hossam (ahmed-hossam-suez-canal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B authenticated


## 📂 Section 4 — Stratified Dataset Subset Builder

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Why stratified, not random?
#
# Your 24 classes have severe imbalance. A naive random 10% sample would give:
#   • car (1240 images)                    → ~124 images  ← still meaningful
#   • motorcycle_truck_accident (37 images) →   ~3 images  ← completely useless
#
# Consequence: rare-class mAP50 becomes random noise, the Bayesian GP learns
# a noisy objective, and the recommended hyperparameters are unreliable.
#
# Strategy:
#   1. Count images per class in the training split.
#   2. Classes below RARE_THRESHOLD → keep ALL their images.
#   3. Common classes → random sample SUBSET_FRAC of their images.
#   4. Validation: always the FULL original val split (honest, comparable mAP50).
# ─────────────────────────────────────────────────────────────────────────────

def _get_dominant_class(lbl_path: Path) -> Optional[int]:
    """Return the most-frequently annotated class in a YOLO-format label file.
    Each image is assigned to one bucket for stratified sampling."""
    counts: Dict[int, int] = {}
    try:
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    c = int(parts[0])
                    counts[c] = counts.get(c, 0) + 1
    except (OSError, ValueError):
        return None
    return max(counts, key=counts.get) if counts else None


def _count_images_per_class(labels_dir: Path) -> Dict[int, int]:
    """Count how many label files contain each class (at least once).
    Used to classify classes as rare vs. common."""
    counts: Dict[int, int] = {c: 0 for c in range(NUM_CLASSES)}
    for lbl in labels_dir.glob("*.txt"):
        seen: set = set()
        try:
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        seen.add(int(parts[0]))
        except (OSError, ValueError):
            continue
        for c in seen:
            counts[c] = counts.get(c, 0) + 1
    return counts


def _find_image_file(stem: str, img_dir: Path) -> Optional[Path]:
    """Find an image file for a given stem with any YOLO-supported extension."""
    for ext in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
        candidate = img_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def build_stratified_subset(seed: int = 42) -> str:
    """Build a stratified 16% training subset and write subset_data.yaml.

    Args:
        seed: Random seed for reproducible sampling.

    Returns:
        str: Absolute path to the generated subset_data.yaml.
    """
    random.seed(seed)

    src     = Path(MERGED_DATASET_PATH)
    src_img = src / "images" / "train"
    src_lbl = src / "labels" / "train"

    # Guard: merged dataset must exist before subsetting
    assert src_img.exists(), (
        f"Training images not found at {src_img}\n"
        "Run the merge cell from your training notebook first."
    )

    # Recreate subset directory from scratch
    dst = Path(SUBSET_PATH)
    if dst.exists():
        shutil.rmtree(dst)
    (dst / "images" / "train").mkdir(parents=True)
    (dst / "labels" / "train").mkdir(parents=True)

    # Classify classes as rare vs. common
    class_counts = _count_images_per_class(src_lbl)
    rare_cls     = {c for c, n in class_counts.items() if n < RARE_THRESHOLD}

    # Print a transparent breakdown so you know what the subset contains
    sep = "-" * 70
    print(sep)
    print(f"  {'Class':<44} {'Images':>6}   Strategy")
    print(sep)
    for c in range(NUM_CLASSES):
        n   = class_counts.get(c, 0)
        tag = "keep ALL  (rare)" if c in rare_cls else f"sample {SUBSET_FRAC:.0%}  (common)"
        print(f"  [{c:2d}] {CLASSES[c]:<44} {n:5d}   → {tag}")
    print(sep)

    # Assign each image to its dominant class for stratified grouping
    class_to_stems: Dict[int, List[str]] = {c: [] for c in range(NUM_CLASSES)}
    for lbl in sorted(src_lbl.glob("*.txt")):
        dom = _get_dominant_class(lbl)
        if dom is not None and _find_image_file(lbl.stem, src_img) is not None:
            class_to_stems[dom].append(lbl.stem)

    # Select which stems go into the subset
    selected_stems: set = set()
    for c in range(NUM_CLASSES):
        stems = class_to_stems[c]
        if not stems:
            continue
        if c in rare_cls:
            selected_stems.update(stems)
        else:
            k = max(1, int(len(stems) * SUBSET_FRAC))
            selected_stems.update(random.sample(stems, k))

    # Copy selected image + label pairs
    copied, skipped = 0, 0
    for stem in sorted(selected_stems):
        img_src = _find_image_file(stem, src_img)
        lbl_src = src_lbl / f"{stem}.txt"
        if img_src is None or not lbl_src.exists():
            skipped += 1
            continue
        shutil.copy(img_src, dst / "images" / "train" / img_src.name)
        shutil.copy(lbl_src, dst / "labels" / "train" / f"{stem}.txt")
        copied += 1

    total_train = sum(1 for _ in src_img.iterdir())
    print(f"\n✅ Subset built → {copied:,} / {total_train:,} training images")
    print(f"   (all rare-class images + {SUBSET_FRAC:.0%} of common-class images)")
    if skipped:
        print(f"   ⚠️  {skipped} stems skipped (missing image or label file)")

    # Write subset_data.yaml
    # NOTE: val = FULL original split — honest, comparable mAP50 across all trials.
    # Using a val subset would introduce noise for rare classes (37-53 val images).
    yaml_data = {
        "path":  SUBSET_PATH,
        "train": "images/train",
        "val":   str(src / "images" / "val"),
        "test":  str(src / "images" / "test"),
        "nc":    NUM_CLASSES,
        "names": CLASSES,
    }
    yaml_path = Path(SUBSET_YAML_PATH)
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_data, f, default_flow_style=None, allow_unicode=True, sort_keys=False)

    print(f"   📄 subset_data.yaml → {yaml_path}")
    return str(yaml_path)


## 🎯 Section 5 — W&B Bayesian Sweep Configuration

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Why only 5 parameters?
#   The Bayesian GP works best when dimensions << trials. With 20 trials,
#   5 parameters is the sweet-spot — covering key dynamics without making the
#   search space too sparse to model reliably.
#
# Parameter ranges — centred around your actual training config values:
#   lr0          : [0.001, 0.02]   — current: 0.01  (log-uniform, ±1 order)
#   lrf          : [0.001, 0.1]    — current: 0.01  (log-uniform, ±1 order)
#   momentum     : [0.60, 0.980]  — current: 0.937 (SGD β1, NOT Adam momentum)
#   momentum     : [0.850, 0.980]  — current: 0.937 (Adam β1, NOT SGD momentum)
#   weight_decay : [0.0001, 0.01]  — current: 0.0005 (log-uniform, wider range)
#   warmup_epochs: [2.0, 5.0]     — current: 3.0    uniform
#
# Metric: "metrics/mAP50(B)" is the exact key Ultralytics W&B callback logs
# per epoch. W&B uses the MAX of this metric across all epochs per trial
# (because goal=maximize), naturally rewarding fast-converging configs.
#
# Hyperband early termination:
#   After epoch 3 (min_iter=3), the bottom half (eta=2) of running trials are
#   killed. For 20 trials this saves ~10 x 2 wasted epochs = 20 ep of compute.
# ─────────────────────────────────────────────────────────────────────────────

SWEEP_CONFIG = {
    "name":   WB_SWEEP_NAME,
    "method": "bayes",           # Gaussian Process surrogate model
    "metric": {
        "name": "metrics/mAP50(B)",  # key logged per epoch by Ultralytics callback
        "goal": "maximize",
    },
    "early_terminate": {
        # Hyperband kills poor runs at epoch 6 of 8, saving ~20% compute
        "type":     "hyperband",
        "min_iter": 6,    # do not kill before epoch 6
        "eta":      2,    # bottom 1/2 killed at each rung (conservative)
    },
    "parameters": {
        "lr0": {
            "distribution": "log_uniform_values",
            "min": 0.001,   # 10x below current (0.01)
            "max": 0.020,   # 2x  above current
        },
        "lrf": {
            "distribution": "log_uniform_values",
            "min": 0.001,   # 10x below current (0.01)
            "max": 0.100,   # 10x above current
        },
        "momentum": {
            # SGD β1 — NOT AdamW momentum.
            "distribution": "uniform",
            "min": 0.600,
            "max": 0.980,
        },
        "weight_decay": {
            "distribution": "log_uniform_values",
            "min": 0.0001,  # 5x  below current (0.0005)
            "max": 0.0100,  # 20x above current — wider for regularization search
        },
        "warmup_epochs": {
            "distribution": "uniform",
            "min": 2.0,
            "max": 5.0,      # current default: 3.0
        },
    },
}


def register_sweep() -> str:
    """Register the Bayesian sweep with W&B and return the sweep_id.
    The returned sweep_id is passed to wandb.agent() to launch the trials."""
    sweep_id = wandb.sweep(sweep=SWEEP_CONFIG, project=WB_PROJECT)
    print(f"✅ Sweep registered")
    print(f"   ID  : {sweep_id}")
    print(f"   URL : https://wandb.ai/<your-entity>/{WB_PROJECT}/sweeps/{sweep_id}")
    return sweep_id


## 🏋️ Section 6 — Sweep Trial Training Function

In [8]:
def sweep_train() -> None:
    """Single W&B sweep trial. Called once per trial by wandb.agent().

    Execution flow:
        1. wandb.init()    → sweep agent injects hyperparameters via wandb.config.
        2. YOLO().train()  → YOLO26n trains for MICRO_EPOCHS on the 16% subset.
                             Ultralytics built-in W&B callback detects the active
                             run and logs metrics/mAP50(B) per epoch to it —
                             this is what the Bayesian GP reads.
        3. wandb.log()     → explicit final summary for the W&B runs table.
        4. wandb.finish()  → always called in finally — closes run on failure too.
        5. GPU cleanup     → frees VRAM and triggers GC before the next trial.

    Key decisions:
        • wandb.init() BEFORE model.train() is mandatory: Ultralytics callback
          attaches to the existing run instead of opening a conflicting new one.
        • save=False  → no weight files saved  (saves ~50 MB × 20 trials on disk).
        • plots=False → skips PNG plots         (saves ~10 s per trial).
        • val=True    → validation MUST run every epoch (it logs metrics/mAP50(B)).
        • On failure, log mAP50=0.0 so W&B records the trial as failed, not missing.
    """
    # Guard: ensure the subset was built before the agent called this function
    assert Path(SUBSET_YAML_PATH).exists(), (
        f"subset_data.yaml not found at {SUBSET_YAML_PATH}.\n"
        "Run build_stratified_subset() (Section 7, Step 2) first."
    )

    run = wandb.init(
        project  = WB_PROJECT,
        group    = WB_GROUP,        # visually groups all 12 trials in W&B UI
        job_type = WB_JOB_TYPE,
        tags     = ["yolo26n", "bayesian-sweep", f"{MICRO_EPOCHS}ep", "MuSGD"],
    )
    cfg        = wandb.config      # hyperparameters injected by the sweep agent
    trial_name = f"trial_{run.id}"

    try:
        model = YOLO(BASE_MODEL)   # reload fine-tuned checkpoint for each trial

        results = model.train(
            data    = SUBSET_YAML_PATH,    # 16% stratified training subset
            epochs  = MICRO_EPOCHS,
            imgsz   = FIXED_IMGSZ,
            batch   = FIXED_BATCH,

            # ── Swept hyperparameters (injected by W&B agent each trial) ────
            optimizer     = FIXED_OPTIMIZER,    # "MuSGD" — locked, not swept
            lr0           = cfg.lr0,
            lrf           = cfg.lrf,
            momentum      = cfg.momentum,       # MuSGD β1 (MuSGD convention)
            weight_decay  = cfg.weight_decay,
            warmup_epochs = cfg.warmup_epochs,
            
            # ── Fixed settings — mirror your actual training logs exactly ────
            cos_lr        = FIXED_COS_LR,       # True — standard LR scheduler
            amp           = FIXED_AMP,           # True  — mixed precision
            patience      = FIXED_PATIENCE,      # 999   — no early stopping
            workers       = FIXED_WORKERS,
            deterministic = FIXED_DETERMINISTIC,
            val           = True,                # needed to log mAP50 per epoch

            # ── Micro-trial I/O flags — skip expensive outputs ───────────────
            plots       = False,    # no PNG plots    (saves ~10s / trial)
            save        = False,    # no .pt weights  (saves ~50MB / trial)
            save_period = 0,        # no periodic weight saves

            project = SWEEP_RESULTS_PATH,
            name    = trial_name,
            verbose = False,        # suppress per-batch YOLO progress bars
        )

        # Extract final-epoch validation metrics from the results object
        m         = results.results_dict
        mAP50     = m.get("metrics/mAP50(B)",     0)
        mAP5095   = m.get("metrics/mAP50-95(B)",  0)
        precision = m.get("metrics/precision(B)", 0)
        recall    = m.get("metrics/recall(B)",    0)
        fitness   = results.fitness  # 0.1 x mAP50 + 0.9 x mAP50-95

        # Log final-epoch summary (visible in W&B runs table for comparison)
        wandb.log({
            "final/mAP50":     mAP50,
            "final/mAP50-95":  mAP5095,
            "final/precision": precision,
            "final/recall":    recall,
            "final/fitness":   fitness,
        })

        print(
            f"  [{trial_name}]  mAP50={mAP50:.4f} | P={precision:.4f} | R={recall:.4f} | "
            f"lr0={cfg.lr0:.5f} | lrf={cfg.lrf:.4f} | "
            f"mom={cfg.momentum:.4f} | wd={cfg.weight_decay:.6f} | "
            f"warmup={cfg.warmup_epochs:.1f}ep"
        )

    except Exception as exc:
        # Log zero so W&B marks this trial as failed (not ignored by the GP model)
        print(f"  [{trial_name}] ❌ Trial failed: {exc}")
        wandb.log({"metrics/mAP50(B)": 0.0, "final/mAP50": 0.0})

    finally:
        # Always close the run and free GPU memory before the next trial starts
        wandb.finish()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()


## 🚀 Section 7 — Run: Build Subset → Register Sweep → Launch Agent

In [ ]:
# ── Step 1: Verify merged dataset exists ──────────────────────────────────────
assert Path(MERGED_DATASET_PATH).exists(), (
    f"Merged dataset not found at {MERGED_DATASET_PATH}\n"
    "Run the dataset merge cell from your training notebook first."
)
n_train = sum(1 for _ in (Path(MERGED_DATASET_PATH) / "images" / "train").iterdir())
n_val   = sum(1 for _ in (Path(MERGED_DATASET_PATH) / "images" / "val").iterdir())
print(f"✅ Dataset ready  →  {n_train:,} train | {n_val:,} val images")

# ── Step 2: Build stratified 10% subset ───────────────────────────────────────
print("\n📦 Building stratified subset...")
subset_yaml = build_stratified_subset(seed=42)

n_subset         = sum(1 for _ in (Path(SUBSET_PATH) / "images" / "train").iterdir())
iters_per_trial  = (n_subset // FIXED_BATCH) * MICRO_EPOCHS
iters_full_run   = (n_train  // FIXED_BATCH) * 50   # estimate for full training

print(f"\n📊 Iteration count estimate:")
print(f"   Micro-trial  :  {n_subset:,} imgs / {FIXED_BATCH} batch x {MICRO_EPOCHS} ep"
      f"  =  {iters_per_trial:,} iters  → AdamW  (auto threshold: 10,000)")
flag = "⚠️  MuSGD if optimizer=auto" if iters_full_run > 10_000 else "AdamW"
print(f"   Full training:  {n_train:,} imgs / {FIXED_BATCH} batch x 50 ep"
      f"  ≈  {iters_full_run:,} iters  → {flag}")
print(f"\n   ➜ We lock optimizer=AdamW in both sweep and final training for consistency.")

# ── Step 3: Register the W&B sweep ────────────────────────────────────────────
print("\n🎯 Registering Bayesian sweep with W&B...")
SWEEP_ID = register_sweep()

# ── Step 4: Launch the sweep agent ────────────────────────────────────────────
# wandb.agent() is synchronous (blocking). It calls sweep_train() NUM_TRIALS
# times sequentially. Each call receives a new hyperparameter config chosen by
# the Bayesian GP based on all previous trial results.
# Estimated time: ~20-40 min on a Kaggle T4 GPU for 20 x 5-epoch trials.
print(f"\n🚀 Launching agent  |  {NUM_TRIALS} trials x {MICRO_EPOCHS} epochs"
      f"  |  ~{NUM_TRIALS * 2}-{NUM_TRIALS * 3} min on T4")
print("-" * 60)

wandb.agent(
    sweep_id = SWEEP_ID,
    function = sweep_train,
    project  = WB_PROJECT,
    count    = NUM_TRIALS,
)

print("-" * 60)
print(f"\n✅ All {NUM_TRIALS} trials complete.")


✅ Dataset ready  →  37,930 train | 5,600 val images

📦 Building stratified subset...
----------------------------------------------------------------------
  Class                                        Images   Strategy
----------------------------------------------------------------------
  [ 0] bus                                           3873   → sample 32%  (common)
  [ 1] bus_bus_accident                               580   → keep ALL  (rare)
  [ 2] bus_object_accident                            608   → keep ALL  (rare)
  [ 3] bus_person_accident                            636   → keep ALL  (rare)
  [ 4] bus_truck_accident                             536   → keep ALL  (rare)
  [ 5] car                                          10182   → sample 32%  (common)
  [ 6] car_bus_accident                               580   → keep ALL  (rare)
  [ 7] car_car_accident                              1939   → keep ALL  (rare)
  [ 8] car_motorcycle_accident                        608   → keep A

wandb: Agent Starting Run: wnlpyiy5 with config:
wandb: 	lr0: 0.01671441327325431
wandb: 	lrf: 0.01397849586358127
wandb: 	momentum: 0.7147944819232068
wandb: 	warmup_epochs: 3.304094677975356
wandb: 	weight_decay: 0.00028910399376522514
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/sweep_subset/subset_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01671441327325431, lrf=0.01397849586358127, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.7147944819232068, mosaic=1.0, multi_scale=0.0, name=trial_wnlpyiy5, nbs=64, nms=False, opset=None

final/fitness,▁
final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/fitness,0.42663
final/mAP50,0.71408
final/mAP50-95,0.42663
final/precision,0.71418
final/recall,0.6521


wandb: Agent Starting Run: z1mwzv61 with config:
wandb: 	lr0: 0.004136951088172563
wandb: 	lrf: 0.0035270858207273703
wandb: 	momentum: 0.7360431419634308
wandb: 	warmup_epochs: 3.899897143320424
wandb: 	weight_decay: 0.0001343943813390286
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/sweep_subset/subset_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.004136951088172563, lrf=0.0035270858207273703, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.7360431419634308, mosaic=1.0, multi_scale=0.0, name=trial_z1mwzv61, nbs=64, nms=False, opset=N

final/fitness,▁
final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/fitness,0.35629
final/mAP50,0.60508
final/mAP50-95,0.35629
final/precision,0.62748
final/recall,0.56451


wandb: Agent Starting Run: 9vue6x8u with config:
wandb: 	lr0: 0.0020364807724967847
wandb: 	lrf: 0.01910899808961042
wandb: 	momentum: 0.724124774022463
wandb: 	warmup_epochs: 3.272072821497094
wandb: 	weight_decay: 0.00021219966773656967
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/sweep_subset/subset_data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=8, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0020364807724967847, lrf=0.01910899808961042, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.724124774022463, mosaic=1.0, multi_scale=0.0, name=trial_9vue6x8u, nbs=64, nms=False, opset=Non